## Gemini -- Updates to prompts!

Push VQA to Google Gemini.

### Docs:

* in theory, there is documentation here: https://ai.google.dev/gemini-api/docs, 
* Here though, I just asked claude again :D

#### Key Points (cp-claude)

* API Key: Get your API key from the [Google Console](https://aistudio.google.com/app/apikey) -- note that you either have to create a new GCP project OR create the API key in an existing project
* Then the rest of the steps is similar for Claude/ChatGPT


First, install anthropic api (also, see .yml file for the environment for this project)

In [1]:
#!pip install google-generativeai pillow

Where are things stored/going to be stored?

In [1]:
dir_api_old = '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/gemini_api_1.5/' #store API results for example data
dir_api = '/Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/gemini_api_1.5/' #store API results for example data
model='gemini-1.5-flash' # test 2.5 later

# dir_api = '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/gemini_api_2.5/' #store API results for example data
# model='gemini-2.5-flash' # test 2.5 
redo_file = '/Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/question_updates.json'

key_file = '/Users/jnaiman/.gemini/key.txt'

# where is VQA dataset?
jsons_dir = '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/jsons/' # directory where jsons created with figure are stored
imgs_dir = '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/imgs/' # where images are stored

# for saving temp images for reading in
tmp_dir = '/Users/jnaiman/Downloads/tmp/' # this might not be used...

img_format = 'png'

In [2]:
import base64
from PIL import Image
import numpy as np
import json
import re
import pickle
import os
from glob import glob
import google.generativeai as genai

# debug
from importlib import reload
from copy import deepcopy


from sys import path
path.append('../')
import utils.llm_utils
reload(utils.llm_utils)
from utils.llm_utils import parse_qa, load_image, get_img_json_pair, parse_for_errors

import time
from utils.plot_qa_utils import get_nplots

In [3]:
# setup
with open(key_file,'r') as f:
    api_key = f.read()

# Configure the API key
genai.configure(api_key=api_key.strip())

In [4]:
model_gemini = genai.GenerativeModel(model)

In [5]:
jsons_to_parse = glob(jsons_dir + '/*.json')
jsons_to_parse[:3]

['/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/jsons/Picture186.json',
 '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/jsons/Picture169.json',
 '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/jsons/Picture56.json']

In [6]:
def send_to_gemini(question_list, image_path, model_gemini,
                    #tmp_dir = '/Users/jnaiman/Downloads/tmp/',
                    test_run = True, 
                    #fac=1.0, 
                    verbose=True, verbose_tokens = False,
                    system_prompt = None,                   
                   #model='gemini-1.5-flash',
                   large_image= False):
    """
    Sends an image + question to Gemini.  Does something different for large file, but might be bad idea.

    system_prompt : set to None to use default from question list
    """
    if system_prompt is None:
        system_prompt = question_list['persona']

    err = False
    #model_gemini = genai.GenerativeModel(model)
    #if verbose: print('Model loaded:', model)
    prompt_save = ''; prompt = ''; response = ''
    if not test_run:
        question = question_list['context'] + " " + question_list['question'] + " " + question_list['format']
        # lowercase the first word, just in case
        question = question.lstrip() # no whitespace
        question = question[0].lower() + question[1:]
        if verbose: print('   on question:',question)
        # Prepare the API request
        prompt = f"I am going to show you an image. Now, {question}"
        prompt_save = f"I am going to show you an image. Now, {question}"
        try:
        #if True:

            # Wait for processing (for video files, you might need to wait longer)
            if large_image:
                # Upload the file to Gemini
                uploaded_file = genai.upload_file(path=image_path)
                import time
                while uploaded_file.state.name == "PROCESSING":
                    time.sleep(1)
                    uploaded_file = genai.get_file(uploaded_file.name)
            else:
                uploaded_file = Image.open(image_path)
            
            # Generate content with the uploaded file
            response = model_gemini.generate_content([prompt, uploaded_file])
            
            if large_image:
                # Clean up - delete the uploaded file
                genai.delete_file(uploaded_file.name)
            
            #return response.text
        
        except Exception as e:
            print('[ERROR]:', str(e))
            err = True

            #return f"Error: {str(e)}"
        
    if not test_run and not err:
        #return response, '',''
        if verbose:
            #print('     response:',response)
            #print('     response.text:',response.text)
            print('     response:', str(response.candidates[0].content.parts[0].text).replace('\n',''))
            #print('     finish reason:', response.finish_reason)
        question_list['Response (raw)'] = deepcopy(response)
        # Get the response from the API
        #answer = deepcopy(str(response.text))
        answer = deepcopy(str(response.candidates[0].content.parts[0].text))
        question_list['raw answer'] = answer
        # also calculate usage
        usage = {
            'input_tokens': response.usage_metadata.prompt_token_count,
            'output_tokens': response.usage_metadata.candidates_token_count,
            'total_tokens': response.usage_metadata.total_token_count,
            'cached_content_tokens': getattr(response.usage_metadata, 'cached_content_token_count', 0)
            }

        question_list['usage'] = usage
        if verbose and verbose_tokens:
            print(f"      - Input tokens: {usage.input_tokens}")
            print(f"      - Output tokens: {usage.output_tokens}")
            print(f"      - Total tokens: {usage.input_tokens + usage.output_tokens}")
        # format answer
        if '```json"' in answer:
            answer_format = answer.split('```json"')[-1].split('\n')[0].replace('\n', '')
        elif '```json\n' in answer:
            answer_format = answer.split('```json\n')[-1].split('\n')[0].replace('\n', '')
        elif '```json' in answer:
            answer_format = answer.split('```json')[-1].split('\n')[0].replace('\n', '')
        else:
            answer_format = answer # just give it a shot!
        #answer.replace("```json\n",'').replace("\n```",'')
        try:
            question_list['Response'] = json.loads(answer_format)
        except:
            question_list['Response'] = answer_format
            question_list['Error'] = 'JSON formatting'
        question_list['Response String'] = answer_format
    elif err:
        question_list['raw answer'] = 'ERROR'
    else:
        question_list['Response'] = 'TEST RUN'
        question_list['Response String'] = 'TEST RUN'
        question_list['Response (raw)'] = 'TEST RUN'
    

    return question_list, prompt_save, system_prompt

In [7]:
with open(redo_file,'r') as f:
    redo_q_list = json.load(f)
    redo_q_list = json.loads(redo_q_list)

questions_redo = []
for q in redo_q_list:
    questions_redo.append(q['old'])

redo_q_list, questions_redo

([{'key': 'aspect ratio',
   'old': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? You are a helpful assistant, please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.',
   'new': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.',
   'new format': 'Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.',
   'new question': 'What is the aspect ratio of this figure?',
   'new context': ''},
  {'key': 'ylabels',
   'old': 'You are a helpful assistant that can analyze images.  What are the x-axis titles for each figure panel? Please format the output as a json as {"ylabels":[]}, where the list is a list of strings of all of the y-axis titles. If there is a single plot, this should be one element in this list, and if th

In [ ]:
reload(utils.llm_utils)
from utils.llm_utils import parse_for_errors

iMax = 2000
verbose = True
test_run = False # run w/o actually pinging openai
restart = False
# set system_prompt to None to default to what is in question list
system_prompt = """You are a helpful assistant that responds only in valid JSON format. Do not include any explanations, reasoning, or text outside of the JSON response."""
#system_prompt = """You must respond with only valid JSON. Start your response immediately with { and end with }. Do not write any text before or after the JSON."""
# temperature=0.1

fac = 1.0

for ijson,json_path in enumerate(jsons_to_parse):
    if ijson >= iMax:
        continue

    print('on', ijson, 'of', min(iMax,len(jsons_to_parse)))

    # get image and base json
    img_path = imgs_dir + json_path.split('/')[-1].removesuffix('.json') + '.' + img_format
    _, img_format_media, base_json, err = get_img_json_pair(img_path, json_path, 
                                                            dir_api, restart=restart,fac=fac,
                                                      tmp_dir=tmp_dir) #, load_image=False)
    if err:
        continue
    if verbose: print('Got image!')

    # ###### create QA ########

    # also load the already-filled json from prior run
    with open(dir_api_old + json_path.split('/')[-1].removesuffix('.json') + '.pickle', 'rb') as f:
        qa_orig, model = pickle.load(f)

    # find ones to rerun
    qa = []; redos = []
    for qa_pair in qa_orig:
        if qa_pair['Q'] in questions_redo:
            Qnew = redo_q_list[questions_redo.index(qa_pair['Q'])]
            print('** REDO:', qa_pair['Q'])
            qaout = deepcopy(qa_pair)
            # empty things
            for k in ['Response', 'raw answer', 'Response String', 'prompt']:
                qaout[k] = ""
            qaout['fac'] = 1.0
            # update question
            qaout['Q'] = Qnew['new']
            qaout['format'] = Qnew['new format']
            qaout['question'] = Qnew['new question']
            qa.append(qaout)
            redos.append(True)
            print('** NEW: ', qaout)
            #import sys; sys.exit()
        else:
            qa.append(deepcopy(qa_pair))
            redos.append(False)

    responses = []; prompts = []; system_prompts = []
    for question_list,rd in zip(qa,redos):
        if rd:
            response, prompt, system_prompt_out = send_to_gemini(question_list, img_path, model_gemini,
                    test_run = test_run, 
                    verbose=verbose,
                    system_prompt = system_prompt)
            responses.append(response)
            question_list['prompt'] = prompt
            question_list['system prompt'] = system_prompt_out
        else:
            responses.append(deepcopy(question_list['Response']))

        #import sys; sys.exit()


    # parse for errors
    print('')
    print('**** Cleaned QA ****')
    qa = parse_for_errors(qa, llm='Gemini')
    #qa = parse_for_errors(qa) # might need to do this again

    # dump to file
    if not test_run:
        with open(dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle', 'wb') as ff:
            pickle.dump([qa, model], ff)
        print('Just saved:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    else:
        print('Would store at:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    #import sys; sys.exit()

on 0 of 200
have file already: /Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/gemini_api_1.5/Picture186.pickle
on 1 of 200
have file already: /Users/jnaiman/Downloads/tmp/iConf2025/tmpMultipanel/LMM_outputs/gemini_api_1.5/Picture169.pickle
on 2 of 200
Got image!
** REDO: You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? You are a helpful assistant, please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.
** NEW:  {'Q': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'A': {'aspect ratio': 1.4103896689207418}, 'Level': 'Level 1', 'type': 'Figure-level questions', 'Response': '', 'persona': 'You are a helpful assistant that can analyze images.', 'context': '', 'question': 'What is the aspect ratio of this figure?', 'format': 'Please format the

/opt/anaconda3/envs/JCDL2025/lib/python3.12/site-packages/PIL/Image.py:3406: DecompressionBombWarning: Image size (116848180 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Got image!
** REDO: You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? You are a helpful assistant, please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.
** NEW:  {'Q': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'A': {'aspect ratio': 2.423609267183984}, 'Level': 'Level 1', 'type': 'Figure-level questions', 'Response': '', 'persona': 'You are a helpful assistant that can analyze images.', 'context': '', 'question': 'What is the aspect ratio of this figure?', 'format': 'Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'Response (raw)': response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
   

/opt/anaconda3/envs/JCDL2025/lib/python3.12/site-packages/PIL/Image.py:3406: DecompressionBombWarning: Image size (111573300 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Got image!
** REDO: You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? You are a helpful assistant, please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.
** NEW:  {'Q': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'A': {'aspect ratio': 1.2292206201761557}, 'Level': 'Level 1', 'type': 'Figure-level questions', 'Response': '', 'persona': 'You are a helpful assistant that can analyze images.', 'context': '', 'question': 'What is the aspect ratio of this figure?', 'format': 'Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'Response (raw)': response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
  

/opt/anaconda3/envs/JCDL2025/lib/python3.12/site-packages/PIL/Image.py:3406: DecompressionBombWarning: Image size (95594400 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Got image!
** REDO: You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? You are a helpful assistant, please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.
** NEW:  {'Q': 'You are a helpful assistant that can analyze images.  What is the aspect ratio of this figure? Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'A': {'aspect ratio': 4.9380480016223665}, 'Level': 'Level 1', 'type': 'Figure-level questions', 'Response': '', 'persona': 'You are a helpful assistant that can analyze images.', 'context': '', 'question': 'What is the aspect ratio of this figure?', 'format': 'Please format the output as a json as {"aspect ratio":""} to store the aspect ratio of the plot.', 'Response (raw)': response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
  

In [ ]:
#question_list

## Look at data

Check out one, if you wanna:

In [ ]:
pickles = glob(dir_api + '*.pickle')
#pickles = glob('/Users/jnaiman/Downloads/tmp/JCDL2025/example_hists/claude_api/*pickle')
pickles[:5]

['/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/gemini_api/Picture169.pickle',
 '/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/gemini_api/Picture186.pickle']

In [ ]:
ifile = 1
with open(pickles[ifile], 'rb') as f:
    qa_in = pickle.load(f)[0]

In [ ]:
qa_in[0]

{'Q': 'You are a helpful assistant that can analyze images.  How many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.',
 'A': {'nrows': 4, 'ncols': 4},
 'Level': 'Level 1',
 'type': 'Figure-level questions',
 'Response': '```json',
 'persona': 'You are a helpful assistant that can analyze images.',
 'context': '',
 'question': 'How many panels are in this figure?',
 'format': 'Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.',
 'raw answer': '```json\n{"nrows": 4, "ncols": 4}\n```',
 'usage': {'input_tokens': 302,
  'output_tokens': 18,
  'total_tokens': 320,
  'cached_content_tokens': 0},
 'Response String': '```json',
 'prompt': 'I am going to show you an image. Now, how many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.',
 'system prompt': 'You are a helpful assis

In [ ]:
#qa_in = parse_for_errors_claude(qa_in, llm='gemini')

Claude outputs reasoning, so we have to do a bit of cleaning from the responses:

In [ ]:
print(pickles[ifile])
print('*********')
for qa_pairs in qa_in:
    print('Prompt:', qa_pairs['prompt'])
    print('  Real A:', qa_pairs['A'])
    print('Claude A:', qa_pairs['Response'])
    print('    input tokens:', qa_pairs['usage']['input_tokens'])
    print('    output tokens:', qa_pairs['usage']['output_tokens'])
    print('')

/Users/jnaiman/Downloads/tmp/iConf2025/visual_qa_multipanel/LMM_outputs/gemini_api/Picture186.pickle
*********
Prompt: I am going to show you an image. Now, how many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.
  Real A: {'nrows': 4, 'ncols': 4}
Claude A: ```json
    input tokens: 302
    output tokens: 18

Prompt: I am going to show you an image. Now, assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot". Examples of plotting styles are "classic" or "ggplot". What is the plot style used in this figure? Please format the output as a json as {"plot style":""} to store the matplotlib plotting style used in the figure.
  Real A: {'plot style': 'fivethirtyeight'}
Claude A: ```json
    input tokens: 339
    output tokens: 12

Prompt: I am going to show you an image. Now, assume this is a figure made with matplotlib in Python. Examples of matplotlib c